In [4]:
import pandas as pd
import numpy as np
import joblib
import time
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

preprocessor = joblib.load("../models/preprocessor.pkl")
X_train_processed = preprocessor.transform(X_train)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [5]:
models = {
    'Ridge': Ridge(alpha=1.0, random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=15,
                                            n_jobs=-1, random_state=42),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=200,
                                                            random_state=42),
    'XGBoost': XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                              n_jobs=-1, random_state=42, verbosity=0),
    'CatBoost': CatBoostRegressor(iterations=200, depth=6, learning_rate=0.05,
                                    random_state=42, verbose=0)
}

In [6]:
results = []

for name, model in models.items():
    start = time.time()
    scores = cross_val_score(model, X_train_processed, y_train,
                               cv=cv, scoring='neg_root_mean_squared_error', n_jobs=1)
    elapsed = time.time() - start

    results.append({
        'Model': name,
        'CV_RMSE_mean': -scores.mean(),
        'CV_RMSE_std': scores.std(),
        'Time_seconds': elapsed
    })
    print(f"{name}: RMSE={-scores.mean():.4f} (+/- {scores.std():.4f}), Time={elapsed:.1f}s")

Ridge: RMSE=0.1195 (+/- 0.0084), Time=0.0s
RandomForest: RMSE=0.1328 (+/- 0.0070), Time=3.3s
HistGradientBoosting: RMSE=0.1281 (+/- 0.0056), Time=6.4s
XGBoost: RMSE=0.1201 (+/- 0.0074), Time=1.2s
CatBoost: RMSE=0.1203 (+/- 0.0099), Time=2.0s


In [7]:
results = []

for name, model in models.items():
    start = time.time()
    scores = cross_val_score(model, X_train_processed, y_train,
                               cv=cv, scoring='neg_root_mean_squared_error', n_jobs=1)
    elapsed = time.time() - start

    results.append({
        'Model': name,
        'CV_RMSE_mean': -scores.mean(),
        'CV_RMSE_std': scores.std(),
        'Time_seconds': elapsed
    })
    print(f"{name}: RMSE={-scores.mean():.4f} (+/- {scores.std():.4f}), Time={elapsed:.1f}s")

Ridge: RMSE=0.1195 (+/- 0.0084), Time=0.0s
RandomForest: RMSE=0.1328 (+/- 0.0070), Time=3.4s
HistGradientBoosting: RMSE=0.1281 (+/- 0.0056), Time=5.9s
XGBoost: RMSE=0.1201 (+/- 0.0074), Time=1.2s
CatBoost: RMSE=0.1203 (+/- 0.0099), Time=1.8s


In [8]:
results_df = pd.DataFrame(results).sort_values('CV_RMSE_mean')
print(results_df)

                  Model  CV_RMSE_mean  CV_RMSE_std  Time_seconds
0                 Ridge      0.119533     0.008398      0.023199
3               XGBoost      0.120121     0.007389      1.176126
4              CatBoost      0.120305     0.009872      1.776748
2  HistGradientBoosting      0.128146     0.005602      5.888687
1          RandomForest      0.132791     0.007040      3.355499
